# Introduction to Difference-in-Differences (DiD) in Python

How much does an after-school tutoring program improve student performance? A school district implemented a new after-school tutoring program in 10 of its 35 high schools. After one year, the average GPA in tutored schools jumped from **60.17** to **96.37** — a staggering **36.20-point** increase. Case closed?

Not quite. Over the same period, GPA also rose in the 25 schools that *did not* receive the program — from **71.22** to **82.10**. Some of the improvement in tutored schools simply reflects a region-wide upward trend. **Difference-in-Differences (DiD)** strips away that common trend and reveals the tutoring program's true causal effect: an **ATT of approximately 25.32 GPA points**.

This tutorial walks through DiD estimation in Python using [PyFixest](https://pyfixest.org/) — a fast, Stata-flavored econometrics package — alongside [Great Tables](https://posit-dev.github.io/great-tables/) for publication-quality output. We use the simulated case study from [Corral and Yang (2024)](https://doi.org/10.1007/s12564-024-09984-9).

**Learning objectives:**

- Explain why **naive before-after comparisons overstate** treatment effects
- Compute **2×2 DiD manually** and via PyFixest's `feols()` function
- Estimate DiD using **multiple equivalent approaches** with a unified formula syntax
- Compare inference under **iid, HC1, CRV1, and CRV3** standard errors
- Build **publication-quality regression tables** with `etable()` and Great Tables
- Estimate and plot **event study models** with `i()` for dynamic treatment effects

## Setup and imports

Install the required packages (not available by default in Google Colab):

In [ ]:
!pip install pyfixest great_tables -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pyfixest as pf
from great_tables import GT, md, style, loc

# Reproducibility
np.random.seed(42)

# Color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
TEAL = "#00d4c8"

| Package | Purpose |
|---------|--------|
| `pyfixest` | Fast fixed-effects estimation with Stata-like formula syntax |
| `great_tables` | Publication-quality HTML/PNG tables from DataFrames |
| `pandas` | Data loading, manipulation, and summary statistics |
| `matplotlib` | Custom figure generation |

## Data loading and exploration

We load the 2×2 dataset directly from GitHub. This Stata `.dta` file contains 35 schools observed across 2 time periods:

In [ ]:
url_did = "https://github.com/quarcs-lab/data-open/raw/master/isds/tutoring_did.dta"
df = pd.read_stata(url_did).astype(float)

print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")

The dataset has **70 observations** (35 schools × 2 periods) and **7 variables**:

- `id` — School identifier (1–35)
- `time` — Time period (1 = pre, 2 = post)
- `treated` — Treatment indicator (1 = received tutoring program)
- `post` — Post-period indicator (1 = after program implementation)
- `txp` — Interaction term (treated × post)
- `gpa` — Outcome: average GPA of low-income students (0–100 scale)
- `female_share` — Share of female students (covariate)

In [ ]:
df.describe().round(2)

In [ ]:
# Crosstab: treatment × time
ct = pd.crosstab(df["treated"], df["post"], margins=True)
ct.index = ["Comparison (0)", "Treated (1)", "Total"]
ct.columns = ["Pre (0)", "Post (1)", "Total"]
ct

We have **10 treated schools** observed in 2 periods (20 observations) and **25 comparison schools** (50 observations). This is a perfectly balanced panel — every school appears exactly once in each period.

### Panel structure visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
schools = sorted(df["id"].unique())
times = sorted(df["time"].unique())

for i, school in enumerate(schools):
    for j, t in enumerate(times):
        row = df[(df["id"] == school) & (df["time"] == t)]
        if len(row) > 0:
            is_treated = row["treated"].values[0] == 1
            is_post = row["post"].values[0] == 1
            if is_treated and is_post:
                color = WARM_ORANGE
            elif is_treated:
                color = "#e8956a"
            else:
                color = STEEL_BLUE
            ax.add_patch(plt.Rectangle((j - 0.4, i - 0.4), 0.8, 0.8,
                                       facecolor=color, edgecolor="white",
                                       linewidth=0.5))

ax.set_xlim(-0.5, len(times) - 0.5)
ax.set_ylim(-0.5, len(schools) - 0.5)
ax.set_xticks(range(len(times)))
ax.set_xticklabels([f"Period {int(t)}" for t in times])
ax.set_yticks(range(0, len(schools), 5))
ax.set_yticklabels([f"School {schools[i]:.0f}" for i in range(0, len(schools), 5)])
ax.set_xlabel("Time Period")
ax.set_ylabel("School ID")
ax.set_title("Panel Structure: 2×2 DiD Design", fontweight="bold")
legend_patches = [
    mpatches.Patch(facecolor=STEEL_BLUE, label="Comparison group"),
    mpatches.Patch(facecolor="#e8956a", label="Treated (pre-program)"),
    mpatches.Patch(facecolor=WARM_ORANGE, label="Treated (post-program)"),
]
ax.legend(handles=legend_patches, loc="upper right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## The problem with naive comparisons

The most intuitive approach to measuring the program's effect is a simple before-after comparison for the treated schools:

In [ ]:
treated_means = df[df["treated"] == 1].groupby("post")["gpa"].mean()
print(f"Pre-program:  {treated_means[0]:.2f}")
print(f"Post-program: {treated_means[1]:.2f}")
print(f"Naive change: {treated_means[1] - treated_means[0]:.2f}")

The naive estimate says the program boosted GPA by **36.20 points**. But this ignores everything else that may have changed over the same period — curriculum reforms, new textbooks, regional economic shifts, or simply students maturing. Any of these factors could drive GPA upward in *all* schools, not just the treated ones.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot([0, 1], [treated_means[0], treated_means[1]], color=WARM_ORANGE,
        marker="o", markersize=10, linewidth=2.5, label="Treated group", zorder=5)
ax.axvline(x=0.5, color="#ff6b6b", linestyle="--", linewidth=1.5, alpha=0.7,
           label="Program starts")
ax.annotate(f"{treated_means[0]:.2f}", (0, treated_means[0]),
            textcoords="offset points", xytext=(-15, 12), fontsize=12,
            color=WARM_ORANGE, fontweight="bold")
ax.annotate(f"{treated_means[1]:.2f}", (1, treated_means[1]),
            textcoords="offset points", xytext=(10, -12), fontsize=12,
            color=WARM_ORANGE, fontweight="bold")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Pre-Program", "Post-Program"])
ax.set_ylabel("Average GPA")
ax.set_title("Naive Before-After Comparison (Treated Group Only)", fontweight="bold")
ax.legend(loc="upper left")
ax.set_ylim(50, 105)
plt.tight_layout()
plt.show()

## The DiD design: using a comparison group

The key insight of DiD is to use the **comparison group** as a mirror for what would have happened to the treated schools *without* the program.

In [ ]:
means = df.groupby(["treated", "post"])["gpa"].mean()
# Round group means for consistent manual arithmetic
pre_control = round(means[(0, 0)], 2)
post_control = round(means[(0, 1)], 2)
pre_treated = round(means[(1, 0)], 2)
post_treated = round(means[(1, 1)], 2)

print(f"Group means:")
print(f"  Comparison Pre:  {pre_control:.2f}")
print(f"  Comparison Post: {post_control:.2f}")
print(f"  Treated Pre:     {pre_treated:.2f}")
print(f"  Treated Post:    {post_treated:.2f}")

counterfactual = pre_treated + (post_control - pre_control)
did_manual = post_treated - counterfactual
print(f"\nCounterfactual: {pre_treated:.2f} + ({post_control:.2f} - {pre_control:.2f}) = {counterfactual:.2f}")
print(f"DiD estimate:   {post_treated:.2f} - {counterfactual:.2f} = {did_manual:.2f}")

The causal effect of the tutoring program is **25.32 GPA points** — not 36.20. The naive approach overstated the effect by **43%** because it attributed the 10.88-point common trend entirely to the program.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([0, 1], [pre_control, post_control], color=STEEL_BLUE, marker="o",
        markersize=10, linewidth=2.5, label="Comparison group", zorder=5)
ax.plot([0, 1], [pre_treated, post_treated], color=WARM_ORANGE, marker="o",
        markersize=10, linewidth=2.5, label="Treated group", zorder=5)
ax.plot([0, 1], [pre_treated, counterfactual], color=TEAL, marker="D",
        markersize=8, linewidth=2, linestyle="--", label="Counterfactual", zorder=4)

mid_y = (post_treated + counterfactual) / 2
ax.annotate("", xy=(1.05, post_treated), xytext=(1.05, counterfactual),
            arrowprops=dict(arrowstyle="<->", color=TEAL, lw=2))
ax.text(1.12, mid_y, f"DiD = {did_manual:.2f}", fontsize=13, color=TEAL,
        fontweight="bold", va="center")

for x, y, label in [(0, pre_control, f"{pre_control:.2f}"),
                     (1, post_control, f"{post_control:.2f}"),
                     (0, pre_treated, f"{pre_treated:.2f}"),
                     (1, post_treated, f"{post_treated:.2f}"),
                     (1, counterfactual, f"{counterfactual:.2f}")]:
    offset = (-35, 8) if x == 0 else (10, 8)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=offset,
                fontsize=11, fontweight="bold")

ax.set_xticks([0, 1])
ax.set_xticklabels(["Pre-Program", "Post-Program"])
ax.set_ylabel("Average GPA")
ax.set_title("Difference-in-Differences: Identifying the Causal Effect", fontweight="bold")
ax.legend(loc="upper left")
ax.set_ylim(50, 105)
ax.set_xlim(-0.15, 1.3)
plt.tight_layout()
plt.show()

### The parallel trends assumption

DiD rests on one critical assumption: **parallel trends**. In the absence of treatment, treated and comparison groups would have followed the *same trajectory* over time. Formally:

$$E[Y_{i,1}(0) - Y_{i,0}(0) \mid D=1] = E[Y_{i,1}(0) - Y_{i,0}(0) \mid D=0]$$

In words: the *change* in potential untreated outcomes is the same for both groups. Think of it like two runners on parallel tracks — they may start at different positions (treated schools have lower baseline GPA), but they run at the same pace.

Note what parallel trends does *not* require: the two groups do not need the same *level* of GPA, only the same *trend*.

## Manual DiD calculation

We can organize the four group means into a 2×2 table and compute the DiD as a *double difference*:

In [ ]:
means_table = df.groupby(["treated", "post"])["gpa"].mean().round(2).unstack()
means_table.index = ["Comparison (0)", "Treated (1)"]
means_table.columns = ["Pre (0)", "Post (1)"]
means_table["Difference"] = means_table["Post (1)"] - means_table["Pre (0)"]
means_table

The DiD formula takes the *difference of differences*:

$$DiD = \Big(E[Y_{i,1} \mid D=1] - E[Y_{i,0} \mid D=1]\Big) - \Big(E[Y_{i,1} \mid D=0] - E[Y_{i,0} \mid D=0]\Big)$$

Plugging in the numbers:

$$DiD = (96.37 - 60.17) - (82.10 - 71.22) = 36.20 - 10.88 = 25.32$$

In [ ]:
did_calc = (post_treated - pre_treated) - (post_control - pre_control)
print(f"DiD = ({post_treated:.2f} - {pre_treated:.2f}) - ({post_control:.2f} - {pre_control:.2f})")
print(f"    = {post_treated - pre_treated:.2f} - {post_control - pre_control:.2f}")
print(f"    = {did_calc:.2f}")

## DiD via regression

### Classical OLS with interaction

The manual calculation is equivalent to an OLS regression with the treatment indicator, time indicator, and their interaction:

$$Y_{it} = \alpha + \beta_1 \text{Treat}_i + \beta_2 \text{Post}_t + \beta_3 (\text{Treat}_i \times \text{Post}_t) + \varepsilon_{it}$$

Where:
- $\alpha$ is the comparison group's pre-period mean (intercept)
- $\beta_1$ captures the baseline difference between groups
- $\beta_2$ captures the common time trend
- $\beta_3$ is the **DiD estimate** — the causal effect of treatment

In [ ]:
fit_ols = pf.feols("gpa ~ treated + post + txp", data=df, vcov="HC1")
fit_ols.summary()

Every coefficient maps directly to our group means:

- **Intercept (71.22)** — Comparison group pre-period mean
- **treated (−11.05)** — Treated schools start 11 points *below* comparison schools
- **post (10.89)** — Common time trend (comparison group's improvement)
- **txp (25.32)** — The DiD estimate, matching our manual calculation

### TWFE with fixed effects

A more flexible approach absorbs school-level and time-level heterogeneity using **two-way fixed effects (TWFE)**. PyFixest uses the `|` pipe syntax to specify absorbed fixed effects:

$$Y_{it} = \beta_3 (\text{Treat}_i \times \text{Post}_t) + \gamma_i + \vartheta_t + \varepsilon_{it}$$

In [ ]:
fit_twfe = pf.feols("gpa ~ txp | id + time", data=df, vcov={"CRV1": "id"})
fit_twfe.summary()

The estimate is unchanged: **25.315**. The formula `"gpa ~ txp | id + time"` is one of PyFixest's key strengths: everything to the left of `|` is estimated, everything to the right is *absorbed*.

### TWFE with covariate

In [ ]:
fit_cov = pf.feols("gpa ~ txp + female_share | id + time", data=df,
                    vcov={"CRV1": "id"})
fit_cov.summary()

Adding `female_share` barely changes the DiD estimate (25.315 → 25.328, a shift of just 0.013). The covariate itself is statistically insignificant (p = 0.714).

### Programmatic access to results

PyFixest provides tidy methods for extracting specific quantities:

In [ ]:
print(f"Coefficient:   {fit_twfe.coef().values[0]:.4f}")
print(f"Std. Error:    {fit_twfe.se().values[0]:.4f}")
print(f"t-statistic:   {fit_twfe.tstat().values[0]:.4f}")
print(f"p-value:       {fit_twfe.pvalue().values[0]:.4f}")
print(f"95% CI:        [{fit_twfe.confint().values[0, 0]:.2f}, {fit_twfe.confint().values[0, 1]:.2f}]")

In [ ]:
fit_twfe.tidy()

### Comparison across specifications

All three specifications produce essentially the same DiD estimate:

In [ ]:
specs = {
    "OLS/HC1": fit_ols,
    "TWFE/CRV1": fit_twfe,
    "TWFE+cov/CRV1": fit_cov,
}
for name, fit in specs.items():
    tidy = fit.tidy()
    txp_row = tidy[tidy.index == "txp"].iloc[0]
    print(f"  {name:18s}: β = {txp_row['Estimate']:.4f}, "
          f"SE = {txp_row['Std. Error']:.4f}, "
          f"95% CI = [{txp_row['2.5%']:.2f}, {txp_row['97.5%']:.2f}]")

## Inference comparison

One of PyFixest's strengths is the ability to quickly compare different inference approaches on the same model:

In [ ]:
vcov_types = {
    "iid": "iid",
    "HC1": "HC1",
    "CRV1": {"CRV1": "id"},
    "CRV3": {"CRV3": "id"},
}

se_results = {}
for label, vcov_spec in vcov_types.items():
    fit_tmp = pf.feols("gpa ~ txp | id + time", data=df, vcov=vcov_spec)
    tidy = fit_tmp.tidy()
    txp_row = tidy[tidy.index == "txp"].iloc[0]
    se_results[label] = {
        "se": txp_row["Std. Error"],
        "tstat": txp_row["t value"],
        "pvalue": txp_row["Pr(>|t|)"],
    }
    print(f"  {label:5s}: SE = {txp_row['Std. Error']:.4f}, "
          f"t = {txp_row['t value']:.2f}, "
          f"p = {txp_row['Pr(>|t|)']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = list(se_results.keys())
ses = [se_results[l]["se"] for l in labels]
colors = [STEEL_BLUE, WARM_ORANGE, TEAL, "#e8956a"]
bars = ax.barh(labels, ses, color=colors, height=0.5)
for bar, se_val in zip(bars, ses):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{se_val:.4f}", va="center", fontsize=11)
ax.set_xlabel("Standard Error of DiD Estimate (txp)")
ax.set_title("Standard Errors Across Inference Methods", fontweight="bold")
ax.set_xlim(0, max(ses) * 1.3)
plt.tight_layout()
plt.show()

**Key takeaway:** inference choice matters less than research design. Standard errors range from 0.585 to 0.637, but all four methods produce p-values that are essentially zero. With only 35 clusters, CRV3 is the safer default.

## Publication-quality tables with etable() and Great Tables

### Stepwise specifications with csw0()

PyFixest's `csw0()` operator estimates multiple specifications in a single call:

In [ ]:
fit_multi = pf.feols("gpa ~ txp + csw0(female_share) | id + time",
                     data=df, vcov={"CRV1": "id"})
models_list = fit_multi.to_list()
print(f"Number of models: {len(models_list)}")
for i, m in enumerate(models_list):
    print(f"\nModel {i+1}:")
    print(m.summary())

### etable() output

In [ ]:
fit_multi.etable()

### Custom Great Tables table

For full control over formatting, we can build a table from the `.tidy()` DataFrames:

In [ ]:
rows = []
for name, fit in [("(1) OLS", fit_ols),
                  ("(2) TWFE", fit_twfe),
                  ("(3) TWFE + Cov", fit_cov)]:
    tidy = fit.tidy()
    txp_row = tidy[tidy.index == "txp"].iloc[0]
    rows.append({
        "Model": name,
        "Estimate": txp_row["Estimate"],
        "Std. Error": txp_row["Std. Error"],
        "t value": txp_row["t value"],
        "p-value": txp_row["Pr(>|t|)"],
        "95% CI Lower": txp_row["2.5%"],
        "95% CI Upper": txp_row["97.5%"],
        "N": fit._N,
    })

gt_df = pd.DataFrame(rows)
(
    GT(gt_df)
    .tab_header(
        title=md("**Table 2: DiD Estimates Across Specifications**"),
        subtitle="Dependent variable: GPA"
    )
    .fmt_number(columns=["Estimate", "Std. Error", "t value",
                         "95% CI Lower", "95% CI Upper"], decimals=3)
    .fmt_number(columns=["p-value"], decimals=4)
    .fmt_integer(columns=["N"])
    .cols_label(**{"95% CI Lower": "CI Lower", "95% CI Upper": "CI Upper"})
    .tab_source_note(
        "Notes: (1) OLS with HC1 robust SE. (2) TWFE with CRV1 "
        "clustered at school level. (3) TWFE with female_share covariate and CRV1."
    )
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(columns="Estimate")
    )
)

### Exporting LaTeX tables for manuscripts

When submitting to academic journals, you need LaTeX-formatted tables rather than HTML or PNG. PyFixest's `etable()` can generate publication-ready LaTeX directly by setting `type="tex"`. The output uses `booktabs` for clean horizontal rules and `threeparttable` for properly aligned footnotes — the standard format expected by most economics and social science journals.

In [ ]:
latex_output = pf.etable(
    [fit_ols, fit_twfe, fit_cov],
    type="tex",
    labels={
        "txp": "Treatment $\\times$ Post",
        "treated": "Treatment",
        "post": "Post",
        "female_share": "Female Share",
        "Intercept": "Constant",
    },
    notes="Standard errors in parentheses. * p<0.05, ** p<0.01, *** p<0.001.",
)
print(latex_output)

To save the table directly to a `.tex` file that you can `\input{}` in your manuscript, use the `file_name` parameter:

In [ ]:
pf.etable(
    [fit_ols, fit_twfe, fit_cov],
    type="tex",
    labels={
        "txp": "Treatment $\\times$ Post",
        "treated": "Treatment",
        "post": "Post",
        "female_share": "Female Share",
        "Intercept": "Constant",
    },
    notes="Standard errors in parentheses. * p<0.05, ** p<0.01, *** p<0.001.",
    file_name="did101_table2.tex",
)
print("Saved: did101_table2.tex")

In your LaTeX manuscript, include the saved file with:

```latex
\begin{table}[htbp]
\centering
\caption{DiD Estimates Across Specifications}
\label{tab:did-results}
\input{did101_table2.tex}
\end{table}
```

Your LaTeX document needs the `booktabs`, `makecell`, and `threeparttable` packages in the preamble:

```latex
\usepackage{booktabs}
\usepackage{makecell}
\usepackage{threeparttable}
```

## Coefficient comparison

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5))
model_names = ["(1) OLS\nHC1", "(2) TWFE\nCRV1", "(3) TWFE+Cov\nCRV1"]
estimates = []
ci_lower = []
ci_upper = []
for fit in [fit_ols, fit_twfe, fit_cov]:
    tidy = fit.tidy()
    txp_row = tidy[tidy.index == "txp"].iloc[0]
    estimates.append(txp_row["Estimate"])
    ci_lower.append(txp_row["2.5%"])
    ci_upper.append(txp_row["97.5%"])

estimates = np.array(estimates)
ci_lower = np.array(ci_lower)
ci_upper = np.array(ci_upper)
y_pos = np.arange(len(model_names))

ax.errorbar(estimates, y_pos, xerr=[estimates - ci_lower, ci_upper - estimates],
            fmt="o", color=TEAL, markersize=10, capsize=6, capthick=2,
            elinewidth=2, ecolor=STEEL_BLUE, zorder=5)

for i, est in enumerate(estimates):
    ax.text(est, i + 0.25, f"{est:.2f}", ha="center", fontsize=11, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(model_names)
ax.set_xlabel("DiD Estimate (txp coefficient)")
ax.set_title("Coefficient Comparison Across Specifications", fontweight="bold")
plt.tight_layout()
plt.show()

The near-identical point estimates and overlapping confidence intervals across all three specifications reinforce that the DiD estimate is robust. The point estimates span a range of just 0.013 GPA points (25.315 to 25.328).

## Event study: dynamic treatment effects

### Loading the event study data

The 2×2 design tells us *whether* the program had an effect, but not *when* the effect kicked in or whether it grew or faded over time. The event study dataset extends the analysis to **8 time periods**:

In [ ]:
url_event = "https://github.com/quarcs-lab/data-open/raw/master/isds/tutoring_didevent.dta"
df_event = pd.read_stata(url_event).astype(float)

print(f"Shape: {df_event.shape}")
print(f"\ntimeToTreat values:")
print(df_event["timeToTreat"].value_counts().sort_index())

The variable `timeToTreat` measures **periods relative to treatment onset** for treated schools: −4 through −1 are pre-treatment periods, 0 through 3 are post-treatment. Untreated schools have `NaN` for this variable.

### The event study specification

The event study model replaces the single `txp` interaction with a full set of **event-time indicators**:

$$Y_{it} = \alpha + \sum_{j=-m}^{q} \theta_j \cdot \text{treat}_{it}(t = k + j) + \gamma_i + \vartheta_t + \varepsilon_{it}$$

Each $\theta_j$ measures the treatment effect at event time $j$:
- **Pre-treatment coefficients** ($j < 0$): Should be near zero if parallel trends holds
- **Post-treatment coefficients** ($j \geq 0$): Capture the dynamic treatment effect

### Estimation with i()

In [ ]:
# Fill NaN timeToTreat (untreated units) with -99 so i() creates a dummy
# that gets absorbed by the school fixed effects
df_event["timeToTreat"] = df_event["timeToTreat"].fillna(-99)

fit_event = pf.feols("gpa ~ i(timeToTreat, ref=-1) | id + time",
                     data=df_event, vcov={"CRV1": "id"})
fit_event.summary()

*Note: Coefficient names are simplified below for readability. PyFixest outputs `C(timeToTreat, contr.treatment(base=-1))[T.X.0]` notation, where X is the relative time period.*

### Event study plot

In [ ]:
import re

tidy_event = fit_event.tidy()
coef_names = tidy_event.index.tolist()
coefs = tidy_event["Estimate"].values
ci_lo = tidy_event["2.5%"].values
ci_hi = tidy_event["97.5%"].values

# Extract numeric time from coefficient names
time_vals = []
for name in coef_names:
    match = re.search(r'T\.([-\d.]+)', name)
    if match:
        time_vals.append(float(match.group(1)))
    else:
        time_vals.append(np.nan)

time_vals = np.array(time_vals)

# Filter out the -99 dummy (untreated placeholder)
valid_mask = (time_vals > -90) & ~np.isnan(time_vals)
time_vals = time_vals[valid_mask]
coefs = coefs[valid_mask]
ci_lo = ci_lo[valid_mask]
ci_hi = ci_hi[valid_mask]

# Add reference period (timeToTreat = -1)
ref_time = -1.0
time_plot = np.concatenate([time_vals[time_vals < ref_time],
                            [ref_time],
                            time_vals[time_vals > ref_time]])
coefs_plot = np.concatenate([coefs[time_vals < ref_time], [0],
                             coefs[time_vals > ref_time]])
ci_lo_plot = np.concatenate([ci_lo[time_vals < ref_time], [0],
                             ci_lo[time_vals > ref_time]])
ci_hi_plot = np.concatenate([ci_hi[time_vals < ref_time], [0],
                             ci_hi[time_vals > ref_time]])

sort_idx = np.argsort(time_plot)
time_plot = time_plot[sort_idx]
coefs_plot = coefs_plot[sort_idx]
ci_lo_plot = ci_lo_plot[sort_idx]
ci_hi_plot = ci_hi_plot[sort_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(time_plot, ci_lo_plot, ci_hi_plot, alpha=0.2, color=STEEL_BLUE)
ax.plot(time_plot, coefs_plot, color=TEAL, marker="o", markersize=8,
        linewidth=2, zorder=5)
ax.axhline(y=0, color="gray", linewidth=0.8, alpha=0.5)
ax.axvline(x=-0.5, color="#ff6b6b", linestyle="--", linewidth=1.5, alpha=0.7,
           label="Treatment onset")

ax.text(-2.5, max(coefs_plot) * 0.85, "Pre-treatment\n(should be ≈ 0)",
        fontsize=11, ha="center", style="italic", color="gray")
ax.text(1.5, max(coefs_plot) * 0.85, "Post-treatment\n(causal effect)",
        fontsize=11, color=WARM_ORANGE, ha="center", style="italic")

ax.set_xlabel("Periods Relative to Treatment")
ax.set_ylabel("Estimated Coefficient")
ax.set_title("Event Study: Dynamic Treatment Effects", fontweight="bold")
ax.set_xticks(time_plot.astype(int))
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

The plot shows a textbook event study pattern:

- **Pre-treatment (t = −4 to −2):** Coefficients are small (0.34, −0.32, 0.59) and statistically insignificant. This is strong evidence supporting the **parallel trends assumption**.

- **Post-treatment (t = 0 to 3):** An immediate, sharp jump to ≈25 points at the moment of treatment, with the effect remaining remarkably stable (24.71 to 25.70). There is **no evidence of fade-out**.

### Event study coefficients table

In [ ]:
event_rows = []
for t, c, lo, hi in zip(time_plot, coefs_plot, ci_lo_plot, ci_hi_plot):
    period_label = f"t = {int(t)}" if t != ref_time else f"t = {int(t)} (ref)"
    sig = ""
    if t == ref_time:
        sig = "(reference)"
    elif lo > 0 or hi < 0:
        sig = "***"
    event_rows.append({
        "Period": period_label,
        "Estimate": round(c, 3),
        "CI Lower": round(lo, 3),
        "CI Upper": round(hi, 3),
        "Significant": sig,
    })

event_df = pd.DataFrame(event_rows)
(
    GT(event_df)
    .tab_header(
        title=md("**Table 4: Event Study Coefficients**"),
        subtitle="Reference period: t = −1 (one period before treatment)"
    )
    .fmt_number(columns=["Estimate", "CI Lower", "CI Upper"], decimals=3)
    .cols_label(**{"CI Lower": "95% CI Lower", "CI Upper": "95% CI Upper"})
    .tab_source_note(
        "Notes: TWFE with school and time FE. CRV1 clustered SE at school level. "
        "*** p < 0.001."
    )
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(columns="Estimate")
    )
)

## Discussion

### Four key findings

1. **The naive before-after comparison overstates the effect by 43%.** The raw change in treated schools is 36.20 GPA points, but 10.88 of these points reflect a common upward trend. DiD correctly attributes only 25.32 points to the program.

2. **The event study confirms no differential pre-trends.** All three pre-treatment coefficients (0.34, −0.32, 0.59) are small, close to zero, and statistically insignificant.

3. **The effect is immediate and sustained.** Post-treatment coefficients range from 24.71 to 25.70, showing no evidence of fade-out.

4. **Inference choice matters less than design.** Standard errors ranged from 0.585 to 0.637 across four inference methods, but all produced t-statistics above 39.

### Caveats

This tutorial uses simulated data designed to illustrate DiD mechanics cleanly. In real applications, you should expect:

- **R-squared below 0.99:** The R² of 0.995 in our TWFE models is unrealistically high.
- **Smaller treatment effects:** A 25-point GPA increase is enormous. Real programs typically produce single-digit effects.
- **Imperfect parallel trends:** Pre-treatment coefficients may not be exactly zero.
- **Staggered treatment timing:** When different units receive treatment at different times, modern DiD estimators (Callaway & Sant'Anna, 2021; Gardner, 2022) address this.

## Summary and takeaways

1. **DiD removes common time trends.** The naive approach overstated the effect by 10.88 GPA points.
2. **Multiple approaches produce one answer.** Three specifications all yield a DiD estimate of 25.32–25.33.
3. **Event studies test parallel trends.** Pre-treatment coefficients (0.34, −0.32, 0.59) are close to zero and insignificant.
4. **The effect is immediate and sustained.** Post-treatment coefficients (24.71–25.70) show no fade-out.
5. **Inference flexibility is a PyFixest strength.** Switching between iid, HC1, CRV1, and CRV3 requires only changing the `vcov` argument.
6. **etable() and Great Tables replace manual table construction.** The `csw0()` operator estimates multiple specifications in one call.

## References

1. Corral, D. & Yang, M. (2024). An introduction to the difference-in-differences design in education policy research. *Asia Pacific Education Review*.
2. Callaway, B. & Sant'Anna, P.H. (2021). Difference-in-differences with multiple time periods. *Journal of Econometrics*, 225(2), 200–230.
3. Goodman-Bacon, A. (2021). Difference-in-differences with variation in treatment timing. *Journal of Econometrics*, 225(2), 254–277.
4. Sun, L. & Abraham, S. (2021). Estimating dynamic treatment effects in event studies with heterogeneous treatment effects. *Journal of Econometrics*, 225(2), 175–199.
5. Fischer, A. & Schar, A. (2024). PyFixest: Fast high-dimensional fixed effects estimation in Python.
6. Gardner, J. (2022). Two-stage differences in differences. Working paper.